# Implementation of the 3rd training pipeline, the "OverEngineered"

    1. Dropping duplicate rows
    2. Applying action mapping
    3. Calculating Bytes and Packets log transformations and log transformed ratios
    4. Applying port bucketing
    5. Log transforming all numeric feautres

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import logging
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer
from sdmetrics.reports.single_table import QualityReport
from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
# logging.basicConfig(level=logging.INFO)

In [3]:
root = Path.cwd().parent 
data_path = root / "Real Data" / "log2.csv"
df = pd.read_csv(data_path)

In [4]:
df = df.drop_duplicates()

action_mapping = {
    'allow': 0,
    'deny': 1,
    'drop': 2,
    'reset-both': 3
}
df['Action'] = df['Action'].map(action_mapping)

In [5]:
# df['Bytes_log'] = np.log1p(df['Bytes Sent'] + df['Bytes Received'])
df['Bytes Ratio'] = np.log1p(df['Bytes Sent'] / (df['Bytes Received'] + 1e-6))  

In [6]:
df['pkts_log'] = np.log1p(df['pkts_sent'] + df['pkts_received'])
df['pkts_ratio'] = np.log1p(df['pkts_sent'] / (df['pkts_received'] + 1e-6))

In [7]:
def bucket_port(port):
    if pd.isna(port):
        return 'unknown'
    elif port <= 1023:
        return 'well_known'
    elif port <= 49151:
        return 'registered'
    else:
        return 'ephemeral'

df['Source Port Category'] = df['Source Port'].apply(bucket_port)
df['Destination Port Category'] = df['Destination Port'].apply(bucket_port)
df['NAT Source Port Category'] = df['NAT Source Port'].apply(bucket_port)
df['NAT Destination Port Category'] = df['NAT Destination Port'].apply(bucket_port)

In [8]:
cols_to_scale = [
    'Bytes',
    'Bytes Sent',
    'Bytes Received',
    'Packets',
    'Elapsed Time (sec)',
    'pkts_sent',
    'pkts_received'
]

df[cols_to_scale] = df[cols_to_scale].apply(lambda x: np.log1p(x))

In [9]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=df)

INFO:sdv.metadata.single_table:Detected metadata:
INFO:sdv.metadata.single_table:{
    "METADATA_SPEC_VERSION": "SINGLE_TABLE_V1",
    "columns": {
        "Source Port": {
            "sdtype": "numerical"
        },
        "Destination Port": {
            "sdtype": "numerical"
        },
        "NAT Source Port": {
            "sdtype": "numerical"
        },
        "NAT Destination Port": {
            "sdtype": "numerical"
        },
        "Action": {
            "sdtype": "categorical"
        },
        "Bytes": {
            "sdtype": "numerical"
        },
        "Bytes Sent": {
            "sdtype": "numerical"
        },
        "Bytes Received": {
            "sdtype": "numerical"
        },
        "Packets": {
            "sdtype": "numerical"
        },
        "Elapsed Time (sec)": {
            "sdtype": "numerical"
        },
        "pkts_sent": {
            "sdtype": "numerical"
        },
        "pkts_received": {
            "sdtype": "numerical"
        }

In [10]:
synthesizer = CTGANSynthesizer(
    metadata=metadata,
    epochs=50,
    verbose=True
)

synthesizer.fit(df)
synthesizer.save('Engineered.pkl')

C:\Users\Карен Оганян\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:144: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\Карен Оганян\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:122: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
INFO:SingleTableSynthesizer:{'EVENT': 'Instance', 'TIMESTAMP': datetime.datetime(2025, 5, 31, 22, 15, 28, 643562), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_7260045972494360bab3e4e5c758793d'}
INFO:SingleTableSynthesizer:{'EVENT': 'Fit', 'TIMESTAMP': datetime.datetime(2025, 5, 31, 22, 15, 28, 643562), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_7260045972494360bab3e4e5c758793d', 'TOTAL NUMBER 

In [11]:
synthetic_data = synthesizer.sample(len(df))

INFO:SingleTableSynthesizer:{'EVENT': 'Sample', 'TIMESTAMP': datetime.datetime(2025, 5, 31, 22, 33, 40, 317304), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_7260045972494360bab3e4e5c758793d', 'TOTAL NUMBER OF TABLES': 1, 'TOTAL NUMBER OF ROWS': 57170, 'TOTAL NUMBER OF COLUMNS': 20}


In [12]:
synthetic_data.to_csv("OverEngineered.csv")

In [13]:
df.to_csv("OverEngineeredREAL.csv")

In [1]:
list(synthetic_data)

NameError: name 'synthetic_data' is not defined